# Financial Markets — Descriptive Analysis

Explores daily market data (S&P 500, Oil, VIX, 10Y Bond, USD Index) and monthly macro indicators
(Unemployment, CPI, Consumer Sentiment, Real Disposable Income, Real GDP) over **June – November 2024**.

Split into two parts:
- **Part 1 — General**: broad trends, correlations, macro overview
- **Part 2 — Events**: market reaction around 4 key political events

In [ ]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from matplotlib.colors import LinearSegmentedColormap

sys.path.insert(0, '..')
from house_style import *
apply_style()

In [ ]:
ROOT   = Path('../../Data/1_Bronze/Financials')
market = pd.read_csv(ROOT / 'market.csv', parse_dates=['Date'])
macros = pd.read_csv(ROOT / 'macros.csv', parse_dates=['DATE'])
market = market.sort_values('Date').reset_index(drop=True)
macros = macros.sort_values('DATE').reset_index(drop=True)

EVENTS = [
    ('Trump shot',                '2024-07-13', REPUBLICAN),
    ('Biden drops out',           '2024-07-21', DEMOCRAT),
    ('Trump-Harris debate',       '2024-09-10', NEUTRAL),
    ('Trump golf attempt',        '2024-09-15', REPUBLICAN),
]

EVENT_PALETTE = {
    'Trump shot':          REPUBLICAN,
    'Biden drops out':     DEMOCRAT,
    'Trump-Harris debate': NEUTRAL,
    'Trump golf attempt':  PALETTE[4],
}

def event_window(df, event_date, days_before=10, days_after=10):
    lo = pd.Timestamp(event_date) - pd.Timedelta(days=days_before)
    hi = pd.Timestamp(event_date) + pd.Timedelta(days=days_after)
    return df[(df['Date'] >= lo) & (df['Date'] <= hi)].copy()

print(f"Market : {len(market)} trading days | {market['Date'].min().date()} → {market['Date'].max().date()}")
print(f"Macros : {len(macros)} months       | {macros['DATE'].min().date()} → {macros['DATE'].max().date()}")


---
## 1 — General

### 1.1 — Data Overview

In [ ]:
print('=== MARKET DATA (daily) ===')
display(market.head())
print(f'\nShape: {market.shape}')
display(market.describe().round(2))

print('\n=== MACROECONOMIC DATA (monthly) ===')
display(macros)

### 1.2 — Market Indicators Over Time

Five daily indicators stacked. Thin line = raw daily values; thick line = 5-day rolling mean.
Dashed vertical lines mark the four political events.

In [ ]:
MARKET_COLS = ['SP500', 'Oil', 'VIX', 'TenYearBond', 'USDIndex']
YLABELS     = ['S&P 500', 'Oil (USD/bbl)', 'VIX', '10Y Yield (%)', 'USD Index']

fig, axes = styled_fig(
    nrows=5, ncols=1, figsize=(14, 18),
    title='US Financial Markets — Jun–Nov 2024',
    sharex=True
)
fig.subplots_adjust(hspace=0.08)

for i, (col, ylabel) in enumerate(zip(MARKET_COLS, YLABELS)):
    ax   = axes[i]
    raw  = market[col]
    roll = raw.rolling(5, min_periods=1).mean()
    ax.plot(market['Date'], raw,  color=PALETTE[0], alpha=0.25, linewidth=1)
    ax.plot(market['Date'], roll, color=PALETTE[0], linewidth=2)
    ax.fill_between(market['Date'], roll, roll.min(), color=PALETTE[0], alpha=0.07)
    for lbl, date, color in EVENTS:
        ax.axvline(
            pd.Timestamp(date), color=color, linestyle='--',
            linewidth=1.2, alpha=0.85,
            label=lbl if i == 0 else '_nolegend_'
        )
    style_ax(ax, ylabel=ylabel)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
axes[-1].xaxis.set_major_locator(mdates.MonthLocator())
plt.setp(axes[-1].get_xticklabels(), rotation=0, ha='center')

handles = [mpatches.Patch(color=c, label=lbl) for lbl, _, c in EVENTS]
axes[0].legend(
    handles=handles, loc='upper left',
    facecolor=BG_PANEL, edgecolor=SPINE_COLOR, labelcolor=TEXT_PRIMARY, fontsize=9
)
plt.tight_layout()
plt.show()

### 1.3 — Correlation Between Market Indicators

Pearson correlation matrix of the five daily market variables.

In [ ]:
corr = market[MARKET_COLS].corr()

fig, ax = styled_fig(figsize=(8, 6), title='Correlation Matrix — Daily Market Indicators')

cmap = LinearSegmentedColormap.from_list(
    'house_div', [REPUBLICAN, BG_PANEL, DEMOCRAT], N=256
)
sns.heatmap(
    corr, ax=ax, annot=True, fmt='.2f', cmap=cmap,
    vmin=-1, vmax=1, center=0,
    linewidths=0.5, linecolor=BG_DARK,
    annot_kws={'size': 10, 'color': TEXT_PRIMARY},
    cbar_kws={'shrink': 0.8},
)
ax.set_facecolor(BG_PANEL)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', color=TEXT_MUTED)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, color=TEXT_MUTED)
cbar = ax.collections[0].colorbar
cbar.ax.tick_params(colors=TEXT_MUTED)
plt.tight_layout()
plt.show()

### 1.4 — S&P 500 vs. Market Volatility (VIX)

Left axis: S&P 500 daily close. Right axis: VIX 10-day rolling mean (shaded).
The inverse relationship between equity levels and volatility is clearly visible around the events.

In [ ]:
fig, ax1 = styled_fig(figsize=(14, 5), title='S&P 500 vs. VIX — Jun–Nov 2024')

ax1.plot(market['Date'], market['SP500'], color=PALETTE[0], linewidth=2, label='S&P 500')
ax1.fill_between(
    market['Date'], market['SP500'], market['SP500'].min(),
    color=PALETTE[0], alpha=0.07
)
style_ax(ax1, ylabel='S&P 500')

ax2      = ax1.twinx()
vix_roll = market['VIX'].rolling(10, min_periods=1).mean()
ax2.plot(market['Date'], vix_roll, color=PALETTE[1], linewidth=2, label='VIX (10d MA)')
ax2.fill_between(market['Date'], vix_roll, 0, color=PALETTE[1], alpha=0.12)
ax2.set_ylabel('VIX', color=PALETTE[1])
ax2.tick_params(axis='y', colors=PALETTE[1])
ax2.spines['right'].set_visible(True)
ax2.spines['right'].set_edgecolor(PALETTE[1])
ax2.set_facecolor('none')

for lbl, date, color in EVENTS:
    ax1.axvline(pd.Timestamp(date), color=color, linestyle='--', linewidth=1.3, alpha=0.85)

y_top = ax1.get_ylim()[1] * 0.99
for lbl, date, color in EVENTS:
    ax1.text(
        pd.Timestamp(date), y_top, f'  {lbl}',
        color=color, fontsize=7.5, va='top', rotation=90, alpha=0.9
    )

ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax1.xaxis.set_major_locator(mdates.MonthLocator())
l1, n1 = ax1.get_legend_handles_labels()
l2, n2 = ax2.get_legend_handles_labels()
ax1.legend(l1 + l2, n1 + n2,
           facecolor=BG_PANEL, edgecolor=SPINE_COLOR, labelcolor=TEXT_PRIMARY)
plt.tight_layout()
plt.show()

### 1.5 — Macroeconomic Indicators

Monthly snapshot of five macro series. Real GDP is quarterly (only 2 data points in the window).

In [ ]:
MACRO_COLS   = ['UnemploymentRate', 'CPIInflation', 'ConsumerSentiment',
                'RealDisposableIncome', 'RealGDP']
MACRO_LABELS = ['Unemployment Rate (%)', 'CPI Inflation Index',
                'Consumer Sentiment', 'Real Disp. Income (Bn USD)', 'Real GDP (Bn USD)']

fig, axes = styled_fig(
    nrows=2, ncols=3, figsize=(18, 9),
    title='US Macroeconomic Indicators — Jun–Nov 2024'
)
fig.subplots_adjust(hspace=0.45, wspace=0.35)

for i, (col, lbl) in enumerate(zip(MACRO_COLS, MACRO_LABELS)):
    ax  = axes[i // 3][i % 3]
    sub = macros[['DATE', col]].dropna()
    if col == 'RealGDP':
        ax.bar(sub['DATE'], sub[col], width=20, color=PALETTE[2],
               alpha=0.85, edgecolor=BG_DARK)
        title_str = lbl + ' (quarterly — 2 obs.)'
    else:
        ax.plot(sub['DATE'], sub[col], color=PALETTE[0], linewidth=2.5,
                marker='o', markersize=7,
                markerfacecolor=BG_PANEL, markeredgecolor=PALETTE[0], markeredgewidth=2)
        ax.fill_between(sub['DATE'], sub[col], sub[col].min(),
                        color=PALETTE[0], alpha=0.07)
        title_str = lbl
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    style_ax(ax, ylabel=lbl, title=title_str)

axes[1][2].set_visible(False)
plt.tight_layout()
plt.show()

---
## 2 — Events

Four key political events during the 2024 US election campaign:

| # | Event | Date | Colour |
|---|-------|------|--------|
| 1 | Trump shot | 2024-07-13 | Red (Republican) |
| 2 | Biden drops out | 2024-07-21 | Blue (Democrat) |
| 3 | Trump-Harris debate | 2024-09-10 | Grey (Neutral) |
| 4 | Trump golf assassination attempt | 2024-09-15 | Orange |

Each event is examined over a **±10 calendar-day window** of daily market data.

### 2.1 — Market Reaction Around Each Event

4 × 2 grid: **left panel** shows S&P 500 (left y) + VIX (right y);
**right panel** shows Oil (left y) + USD Index (right y).
The vertical dashed line marks the exact event date.

In [ ]:
fig, axes = styled_fig(
    nrows=4, ncols=2, figsize=(16, 22),
    title='Market Reactions Around Key Political Events'
)
fig.subplots_adjust(hspace=0.48, wspace=0.32)

for row, (lbl, date, event_color) in enumerate(EVENTS):
    ev  = pd.Timestamp(date)
    win = event_window(market, date, days_before=10, days_after=10)

    ax_l  = axes[row][0]
    ax_l2 = ax_l.twinx()
    ax_l.plot(win['Date'], win['SP500'], color=PALETTE[0], linewidth=2, label='S&P 500')
    ax_l.fill_between(win['Date'], win['SP500'], win['SP500'].min(),
                      color=PALETTE[0], alpha=0.10)
    ax_l.axvline(ev, color=event_color, linestyle='--', linewidth=1.8, alpha=0.9)
    style_ax(ax_l, ylabel='S&P 500')
    ax_l.set_title(f'{lbl}  ({date})', color=event_color, fontweight='bold')
    ax_l.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
    ax_l.xaxis.set_major_locator(mdates.DayLocator(interval=5))
    plt.setp(ax_l.get_xticklabels(), rotation=45, ha='right')
    ax_l2.plot(win['Date'], win['VIX'], color=PALETTE[1], linewidth=2,
               linestyle='-.', label='VIX', alpha=0.9)
    ax_l2.set_ylabel('VIX', color=PALETTE[1])
    ax_l2.tick_params(axis='y', colors=PALETTE[1])
    ax_l2.spines['right'].set_visible(True)
    ax_l2.spines['right'].set_edgecolor(PALETTE[1])
    ax_l2.set_facecolor('none')
    ll, nl   = ax_l.get_legend_handles_labels()
    ll2, nl2 = ax_l2.get_legend_handles_labels()
    ax_l.legend(ll + ll2, nl + nl2, fontsize=8, loc='upper left',
                facecolor=BG_PANEL, edgecolor=SPINE_COLOR, labelcolor=TEXT_PRIMARY)

    ax_r  = axes[row][1]
    ax_r2 = ax_r.twinx()
    ax_r.plot(win['Date'], win['Oil'], color=PALETTE[4], linewidth=2, label='Oil')
    ax_r.fill_between(win['Date'], win['Oil'], win['Oil'].min(),
                      color=PALETTE[4], alpha=0.10)
    ax_r.axvline(ev, color=event_color, linestyle='--', linewidth=1.8, alpha=0.9)
    style_ax(ax_r, ylabel='Oil (USD/bbl)')
    ax_r.set_title(f'{lbl}  ({date})', color=event_color, fontweight='bold')
    ax_r.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
    ax_r.xaxis.set_major_locator(mdates.DayLocator(interval=5))
    plt.setp(ax_r.get_xticklabels(), rotation=45, ha='right')
    ax_r2.plot(win['Date'], win['USDIndex'], color=PALETTE[5], linewidth=2,
               linestyle='-.', label='USD Index', alpha=0.9)
    ax_r2.set_ylabel('USD Index', color=PALETTE[5])
    ax_r2.tick_params(axis='y', colors=PALETTE[5])
    ax_r2.spines['right'].set_visible(True)
    ax_r2.spines['right'].set_edgecolor(PALETTE[5])
    ax_r2.set_facecolor('none')
    lr, nr   = ax_r.get_legend_handles_labels()
    lr2, nr2 = ax_r2.get_legend_handles_labels()
    ax_r.legend(lr + lr2, nr + nr2, fontsize=8, loc='upper left',
                facecolor=BG_PANEL, edgecolor=SPINE_COLOR, labelcolor=TEXT_PRIMARY)

plt.tight_layout()
plt.show()

### 2.2 — Indexed Performance Around Events

Each event window is indexed to **100 on the event day (t = 0)**.
This normalises away level differences and makes relative reactions directly comparable.
Left: S&P 500 ±7 calendar days. Right: VIX ±7 calendar days.

In [ ]:
def index_at_event(df, col, event_date_str, days_before=7, days_after=7):
    ev      = pd.Timestamp(event_date_str)
    lo      = ev - pd.Timedelta(days=days_before)
    hi      = ev + pd.Timedelta(days=days_after)
    win     = df[(df['Date'] >= lo) & (df['Date'] <= hi)].copy().reset_index(drop=True)
    ev_pos  = (win['Date'] - ev).abs().idxmin()
    base    = win.loc[ev_pos, col]
    indexed  = (win[col] / base) * 100
    rel_days = (win['Date'] - ev).dt.days
    return rel_days.values, indexed.values

fig, axes = styled_fig(
    nrows=1, ncols=2, figsize=(16, 6),
    title='Indexed Performance Around Events  (event day = 100)'
)

for ax_i, (col, col_label) in enumerate([('SP500', 'S&P 500'), ('VIX', 'VIX')]):
    ax = axes[ax_i]
    for lbl, date, _ in EVENTS:
        color = EVENT_PALETTE[lbl]
        rel, vals = index_at_event(market, col, date)
        ax.plot(rel, vals, color=color, linewidth=2.5, label=lbl,
                marker='o', markersize=4,
                markerfacecolor=BG_PANEL, markeredgecolor=color, markeredgewidth=1.5)
    ax.axhline(100, color=TEXT_MUTED, linestyle='--', linewidth=1, alpha=0.6)
    ax.axvline(0,   color=TEXT_MUTED, linestyle=':',  linewidth=1, alpha=0.5)
    style_ax(ax,
             xlabel='Days relative to event',
             ylabel='Indexed value (event day = 100)',
             title=col_label)
    ax.legend(facecolor=BG_PANEL, edgecolor=SPINE_COLOR,
              labelcolor=TEXT_PRIMARY, fontsize=9)

plt.tight_layout()
plt.show()

### 2.3 — Market Impact Summary

Percentage change from the event day to **t+1** and **t+5** trading days for all five indicators.
Green = positive, red = negative.

In [ ]:
def pct_change_from_event(df, col, event_date_str, offset_days):
    ev       = pd.Timestamp(event_date_str)
    fwd_ev   = df[df['Date'] >= ev]
    if fwd_ev.empty:
        return np.nan
    base_val = fwd_ev.iloc[0][col]
    target   = ev + pd.Timedelta(days=offset_days)
    fwd_tgt  = df[df['Date'] >= target]
    if fwd_tgt.empty:
        return np.nan
    tgt_val  = fwd_tgt.iloc[0][col]
    return (tgt_val - base_val) / base_val * 100

TRACK_COLS = ['SP500', 'VIX', 'Oil', 'TenYearBond', 'USDIndex']
rows = []
for lbl, date, _ in EVENTS:
    row = {'Event': lbl, 'Date': date}
    for col in TRACK_COLS:
        row[f'{col} +1d'] = pct_change_from_event(market, col, date, 1)
        row[f'{col} +5d'] = pct_change_from_event(market, col, date, 5)
    rows.append(row)

summary = pd.DataFrame(rows).set_index(['Event', 'Date'])

def color_pct(val):
    if not isinstance(val, float) or np.isnan(val):
        return ''
    return 'color: #5cde9f' if val > 0 else 'color: #e6524d'

(summary
 .style
 .format('{:+.2f}%')
 .applymap(color_pct)
 .set_caption('% change from event day to t+1 and t+5 trading days')
)